# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List all record sets and their @ids
print("Record sets in dataset (by @id):")
recsets = list(dataset.metadata.record_sets)
for rs in recsets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    # List fields for each record set
    if 'fields' in rs and rs['fields']:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    - @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Choose record sets by their @id from the above listing
# For the FAIR^2 dataset, the main table is typically called by an explicit @id, e.g., 'dv:dataset_records' or similar.
# Let's retrieve all record set @ids dynamically:

record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded record set: {record_set_id} (Shape: {dataframes[record_set_id].shape})")
        else:
            print(f"Record set {record_set_id} contains no records.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, choose the first record set for further EDA if available
if record_sets:
    main_record_set = record_sets[0]
    print(f"Columns available in {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalization, and grouping. All filtering and grouping references use explicit field `@id`s for reproducibility.


In [ ]:
# Select a numeric field @id for analysis
# List all numeric fields (float/int) using metadata
import numpy as np

numeric_fields = []
for field in dataset.metadata.record_sets[0].get('fields', []):
    dtype = field.get('dataType', '').lower()
    if any([t in dtype for t in ['float', 'number', 'int']]):
        numeric_fields.append(field['@id'])
print(f"Numeric field @ids in {main_record_set}: {numeric_fields}")

# Use the first numeric field available (or fallback to a known numeric col)
if numeric_fields:
    numeric_field = numeric_fields[0]
    # Some field @ids may not correspond 1:1 to DataFrame column names; check for both
    df = dataframes[main_record_set]
    df_cols = list(df.columns)
    if numeric_field in df_cols:
        pass
    else:
        # Try to find similar column (since sometimes field @id is used as col name, or just the tail after colon)
        possible = [c for c in df_cols if c.split(":")[-1] == numeric_field.split(":")[-1]]
        if possible:
            numeric_field = possible[0]
else:
    raise Exception('No numeric field available for EDA.')

# Filter for records above a threshold and normalize
threshold = df[numeric_field].astype(float).mean() if df[numeric_field].dtype != np.float64 else 10
filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalization (Z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely group field (e.g., sample/condition/description)
group_field_candidates = [c for c in df.columns if 'description' in c.lower() or 'sample' in c.lower() or 'condition' in c.lower()]
group_field = group_field_candidates[0] if len(group_field_candidates) > 0 else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing means):")
    display(grouped_df.head())

## 5. Visualization
Visualize data: plot distribution of a numeric field and compare group averages if applicable.


In [ ]:
# Visualize numeric field distribution (histogram)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].astype(float), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If available, plot group means
if group_field:
    plt.figure(figsize=(10,4))
    group_means = df.groupby(group_field)[numeric_field].mean().dropna()
    group_means.plot(kind='bar')
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to load the FAIR² dataset, explored available record sets and fields by their `@id`, filtered and normalized a numeric attribute, and visualized its distribution. This workflow can be extended to deeper analyses, additional field comparisons, or integrated into automated pipelines. The use of `@id` referencing ensures that processing steps remain consistent with the Croissant schema.
